# 02 — Population par carreau de 200 m

**Rôle dans le pipeline :** charger les carreaux INSEE, garder le périmètre étudié et placer chaque carreau en latitude/longitude, le système des GTFS.

| Lit | Produit |
|---|---|
| `data/bronze/insee/carreaux_200m_reun.csv` (Filosofi 2021) | `data/silver/carreaux.parquet` |

**Source :** INSEE, Filosofi 2021, carreaux de 200 m — https://www.insee.fr/fr/statistiques/8735162. Source **fiscale** : proche du recensement, mais pas identique.

In [1]:
import pandas as pd
from pyproj import Transformer
from parametres import INSEE_CSV, SILVER, COMMUNES, TERRITOIRE, CRS_CARREAUX, TAILLE_CARREAU_M

SILVER.mkdir(parents=True, exist_ok=True)

### Contrôles sur le fichier complet

Valeurs attendues (constatées lundi sur Fabric) : **14 773 carreaux**, **34 colonnes**, **884 164 habitants**.

`i_est_200 = 1` signale un carreau dont les valeurs sont **estimées** par l'INSEE (moins de 11 ménages). On mesure leur part en **carreaux**, puis en **habitants** : c'est la seconde qui compte pour la fiabilité des résultats (attendu : 53 % des carreaux, mais seulement 9,5 % des habitants).

In [2]:
df_pop = pd.read_csv(INSEE_CSV)
print("Taille (carreaux, colonnes) :", df_pop.shape)
print("Population totale :", round(df_pop["ind"].sum()))

imputes = df_pop["i_est_200"] == 1
print(f"Carreaux imputés : {imputes.mean():.0%} des carreaux, "
      f"{df_pop.loc[imputes, 'ind'].sum() / df_pop['ind'].sum():.1%} des habitants")

Taille (carreaux, colonnes) : (14773, 34)
Population totale : 884164
Carreaux imputés : 53% des carreaux, 9.5% des habitants


### Périmètre étudié

`lcog_geo` contient le code de la commune. Un carreau à cheval sur deux communes porte plusieurs codes collés ; **le premier est la commune principale**, d'où `str[:5]`.

Filtrer par commune ne contredit pas la règle du brief (« pas de règle basée sur le nom d'une commune ») : ici, on **choisit le périmètre**, on ne touche pas au score. Pour le Nord, il suffit de changer `COMMUNES` dans `parametres.py`.

In [3]:
df_pop["commune"] = df_pop["lcog_geo"].astype(str).str[:5]
carreaux = df_pop[df_pop["commune"].isin(COMMUNES)].copy()

print(f"{TERRITOIRE} : {len(carreaux)} carreaux, {round(carreaux['ind'].sum())} habitants")
print(carreaux.groupby("commune")["ind"].sum().round().rename(index=COMMUNES).sort_values(ascending=False))

Ouest (TCO) : 3691 carreaux, 219406 habitants
commune
Saint-Paul       105576.0
La Possession     36678.0
Saint-Leu         36032.0
Le Port           34054.0
Trois-Bassins      7066.0
Name: ind, dtype: float64


### Centre de chaque carreau, en latitude/longitude

L'identifiant (ex. `CRS2975RES200mN7634200E359400`) donne le **coin bas-gauche** du carreau en mètres : `N` = nord, `E` = est. On ajoute une demi-taille de carreau (100 m) pour obtenir le **centre**, puis on convertit de `EPSG:2975` (mètres) vers `EPSG:4326` (GPS). `always_xy=True` impose l'ordre (x = est, y = nord) et évite l'inversion classique latitude/longitude.

**Contrôle :** les carreaux doivent couvrir la même zone que les arrêts (lat ≈ -21,25 à -20,90 ; lon ≈ 55,22 à 55,44).

In [4]:
coords = carreaux["idcar_200m"].str.extract(r"N(\d+)E(\d+)").astype(int)
carreaux["y_m"] = coords[0] + TAILLE_CARREAU_M / 2
carreaux["x_m"] = coords[1] + TAILLE_CARREAU_M / 2

transfo = Transformer.from_crs(CRS_CARREAUX, "EPSG:4326", always_xy=True)
carreaux["lon"], carreaux["lat"] = transfo.transform(carreaux["x_m"].values, carreaux["y_m"].values)

carreaux[["lat", "lon"]].describe().round(3)

,lat,lon
count,3691.000,3691.000
mean,-21.060,55.303
std,0.081,0.033
min,-21.248,55.217
25%,-21.117,55.285
50%,-21.051,55.303
75%,-20.998,55.325
max,-20.898,55.436


In [5]:
carreaux.to_parquet(SILVER / "carreaux.parquet", index=False)
print("Écrit :", SILVER / "carreaux.parquet", "|", len(carreaux), "carreaux")

Écrit : C:\Users\nicol\Documents\Epitech\hackathon-citiz\data\silver\carreaux.parquet | 3691 carreaux
